In [18]:
import numpy as np
import pandas as pd

In [21]:
d1= pd.read_csv('../data/previous_application.csv')
d2= pd.read_csv('../data/installments_payments.csv')
d3= pd.read_csv('../data/credit_card_balance.csv')
d4= pd.read_csv('../data/application_train.csv')
d5= pd.read_csv('../data/bureau.csv')

In [22]:
bureau_features = d5.groupby('SK_ID_CURR').agg(
    TOTAL_PAST_LOANS = ('SK_ID_BUREAU', 'count'),
    TOTAL_PAST_DEFAULTS = ('CREDIT_DAY_OVERDUE', lambda x: (x > 0).sum()),
    AVG_DAYS_OVERDUE = ('CREDIT_DAY_OVERDUE', 'mean'),
    AVG_DEBT_REMAINING = ('AMT_CREDIT_SUM_DEBT', 'mean'),
    TOTAL_CREDIT_SUM = ('AMT_CREDIT_SUM', 'sum')
).reset_index()
 

In [23]:
cols = [
    'SK_ID_CURR',
    'TARGET',
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'DAYS_EMPLOYED',
    'DAYS_BIRTH',
    'CNT_CHILDREN',
    'CODE_GENDER',
    'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS',
    'FLAG_OWN_CAR',
    'FLAG_OWN_REALTY',
    'REGION_POPULATION_RELATIVE',
    'DAYS_REGISTRATION',
    'AMT_GOODS_PRICE',
    'NAME_INCOME_TYPE',
    'ORGANIZATION_TYPE'
]
 
df = d4[cols].copy()
 
df = df.merge(bureau_features, on='SK_ID_CURR', how='left')

In [24]:
d1_features =d1.groupby('SK_ID_CURR').agg(
    PREV_APP_COUNT=('SK_ID_PREV','count'),
 
    PREV_APPROVED=('NAME_CONTRACT_STATUS',
                   lambda x: (x=='Approved').sum()),
 
    PREV_REFUSED=('NAME_CONTRACT_STATUS',
                  lambda x: (x=='Refused').sum()),
 
    AVG_PREV_CREDIT=('AMT_CREDIT','mean'),
 
    MAX_PREV_CREDIT=('AMT_CREDIT','max'),
 
    AVG_PREV_ANNUITY=('AMT_ANNUITY','mean'),
 
    AVG_PREV_INSTALLMENTS=('CNT_PAYMENT','mean')
).reset_index()

In [25]:
df = df.merge(d1_features, on='SK_ID_CURR', how='left')

In [26]:
d2['DAYS_LATE'] = (
    d2['DAYS_ENTRY_PAYMENT']
    - d2['DAYS_INSTALMENT']
)
 
d2['PAYMENT_RATIO'] = (
    d2['AMT_PAYMENT']
    / d2['AMT_INSTALMENT']
)

In [27]:
d2_features = d2.groupby('SK_ID_CURR').agg(
    AVG_DAYS_LATE=('DAYS_LATE','mean'),
 
    MAX_DAYS_LATE=('DAYS_LATE','max'),
 
    NUM_LATE_PAYMENTS=(
        'DAYS_LATE',
        lambda x:(x>0).sum()
    ),
 
    AVG_PAYMENT_RATIO=(
        'PAYMENT_RATIO',
        'mean'
    ),
 
    MIN_PAYMENT_RATIO=(
        'PAYMENT_RATIO',
        'min'
    )
).reset_index()

In [28]:
df = df.merge(d2_features, on='SK_ID_CURR', how='left')

In [29]:
d3['UTILIZATION'] = (
    d3['AMT_BALANCE']
    / d3['AMT_CREDIT_LIMIT_ACTUAL']
)

In [30]:
d3_features = d3.groupby('SK_ID_CURR').agg(
    AVG_UTILIZATION=('UTILIZATION','mean'),
 
    MAX_UTILIZATION=('UTILIZATION','max'),
 
    HIGH_UTILIZATION_RATE=('UTILIZATION', lambda x: (x > 0.75).mean()),
 
    AVG_DPD=('SK_DPD','mean'),
 
    MAX_DPD=('SK_DPD','max'),
 
    AVG_BALANCE=('AMT_BALANCE','mean')
).reset_index()

In [31]:
df = df.merge(d3_features, on='SK_ID_CURR', how='left')
df = df.drop('SK_ID_CURR', axis=1)

In [32]:
df['DEBT_TO_INCOME'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
 
df['ANNUITY_TO_INCOME'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
 
df['CREDIT_TO_GOODS'] = df['AMT_CREDIT'] / df['AMT_GOODS_PRICE']
 
df['AGE_YEARS'] = (-df['DAYS_BIRTH'] / 365).round(1)

df['EMPLOYMENT_YEARS'] = (-df['DAYS_EMPLOYED'] / 365).round(1)
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, None)
df['EMPLOYMENT_YEARS'] = df['EMPLOYMENT_YEARS'].replace(1000.7, None)
 
df['REGISTRATION_YEARS'] = (-df['DAYS_REGISTRATION'] / 365).round(1)
  
df['EMPLOYMENT_TO_AGE'] = df['EMPLOYMENT_YEARS'] / df['AGE_YEARS']
 
df['CREDIT_TO_AGE'] = df['AMT_CREDIT'] / df['AGE_YEARS']
 
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / (df['CNT_CHILDREN'] + 1)
 
df['ANNUITY_TO_GOODS'] = df['AMT_ANNUITY'] / df['AMT_GOODS_PRICE']
 
df['LOANS_PER_YEAR'] = df['TOTAL_PAST_LOANS'] / df['AGE_YEARS']
 
df['TOTAL_DEBT_TO_INCOME'] = df['AVG_DEBT_REMAINING'] / df['AMT_INCOME_TOTAL']
df['PAST_DEFAULT_RATE'] = (
    df['TOTAL_PAST_DEFAULTS'] / df['TOTAL_PAST_LOANS']
).fillna(0)
df['PREV_APPROVAL_RATE'] = (
    df['PREV_APPROVED'] / df['PREV_APP_COUNT']
).fillna(0)
df['PREV_REFUSAL_RATE'] = (
    df['PREV_REFUSED'] / df['PREV_APP_COUNT']
).fillna(0)

df = df.replace([np.inf, -np.inf], np.nan)

df = df.drop(['DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION'], axis=1)
 
print("New shape:", df.shape)

C:\Users\matis\AppData\Local\Temp\ipykernel_19544\3629603526.py:36: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace([np.inf, -np.inf], np.nan)


New shape: (307511, 52)


In [33]:
numeric_cols = df.select_dtypes(include='number').columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
 
print("Missing values remaining:")
print(df.isnull().sum().sum())

Missing values remaining:
0


In [34]:
from sklearn.preprocessing import LabelEncoder


In [35]:
 
text_cols = df.select_dtypes(include='object').columns
print("Text columns:", list(text_cols))
 
le = LabelEncoder()
for col in text_cols:
    df[col] = le.fit_transform(df[col].astype(str))
 
print(df.shape)
df.head()

Text columns: ['CODE_GENDER', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_INCOME_TYPE', 'ORGANIZATION_TYPE']
(307511, 52)


,TARGET,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,CNT_CHILDREN,CODE_GENDER,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,FLAG_OWN_CAR,FLAG_OWN_REALTY,...,REGISTRATION_YEARS,EMPLOYMENT_TO_AGE,CREDIT_TO_AGE,INCOME_PER_PERSON,ANNUITY_TO_GOODS,LOANS_PER_YEAR,TOTAL_DEBT_TO_INCOME,PAST_DEFAULT_RATE,PREV_APPROVAL_RATE,PREV_REFUSAL_RATE
0,1,202500.0,406597.5,24700.5,0,1,4,3,0,1,...,10.0,0.065637,15698.745174,202500.0,0.070372,0.308880,0.242747,0.0,1.000000,0.000000
1,0,270000.0,1293502.5,35698.5,0,0,1,1,0,0,...,3.2,0.071895,28180.882353,270000.0,0.031606,0.087146,0.000000,0.0,1.000000,0.000000
2,0,67500.0,135000.0,6750.0,0,1,4,3,1,1,...,11.7,0.011494,2586.206897,67500.0,0.050000,0.038314,0.000000,0.0,1.000000,0.000000
3,0,135000.0,312682.5,29686.5,0,0,4,0,0,1,...,26.9,0.159309,6001.583493,135000.0,0.099955,0.103734,0.297350,0.0,0.555556,0.111111
4,0,121500.0,513000.0,21865.5,0,1,4,3,0,1,...,11.8,0.152015,9395.604396,121500.0,0.042623,0.018315,0.000000,0.0,1.000000,0.000000


In [36]:
df.to_csv('../data/added_features_clean_data.csv', index=False)
